# SCIO-Bench: Anomaly Detection for Off-Grid Solar-Battery IoT

**Universitas Jenderal Soedirman — Karel Tsalasatir Riyan — April 2026**

This notebook is the reproducible wrapper for the entire SCIO-Bench experiment pipeline. The framework comprises 14 phases (0 to 13), fully modularized into Python packages under `src/`.

In [9]:
import os
# Change the working directory to the parent folder (project root)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/karel/code/SCIO-Bench


## Phase 1 to 4: Data Engineering Pipeline

- Downloads Kaggle dataset
- Augments synthetic variables (SOC%, V, I)
- Injects 7 anomaly profiles (Physical & Cyber)
- Extracts rolling features and performs 70/15/15 chronological split

In [10]:
!python -m src.data.download
!python -m src.data.preprocess
!python -m src.data.augmentation
!python -m src.data.anomaly_injection
!python -m src.data.feature_engineering

<frozen runpy>:128: RuntimeWarning: 'src.data.download' found in sys.modules after import of package 'src.data', but prior to execution of 'src.data.download'; this may result in unpredictable behaviour
[download] Dataset already present in /home/karel/code/SCIO-Bench/data/raw/ — skipping download.
<frozen runpy>:128: RuntimeWarning: 'src.data.preprocess' found in sys.modules after import of package 'src.data', but prior to execution of 'src.data.preprocess'; this may result in unpredictable behaviour
[preprocess] Processing Plant 1 …
  Gen rows: 68778 | Weather rows: 3182
  After merge+resample: 1632 rows (2020-05-15 → 2020-06-17)
  ✓ Plant 1 ready: (1632, 9) | Columns: ['timestamp', 'device_id', 'mppt_w', 'prod_wh', 'temp_c', 'irradiance', 'ambient_temp_c', 'daily_yield_kwh', 'ac_power_kw']
[preprocess] Processing Plant 2 …
  Gen rows: 67698 | Weather rows: 3259
  After merge+resample: 1632 rows (2020-05-15 → 2020-06-17)
  ✓ Plant 2 ready: (1632, 9) | Columns: ['timestamp', 'device_i

## Phase 5 to 7: Anomaly Detection (Level 1)

We benchmark Method A (Rule-Based), Method B (Isolation Forest, LOF) and Method C (LSTM Autoencoder).

In [11]:
!python -m src.models.rule_based
!python -m src.models.classical_ml
!python -m src.models.lstm_autoencoder

[rule_based] Loading splits …
[rule_based] Train=2,284 Val=489 Test=491

[rule_based] Tuning k on validation set …
[rule_based] Thresholds calibrated (k=2.0):
  R2 power_delta :     0.06 W
  R3 batt_drain  :     0.34 %/tick
  R5 volt_jump   :   0.0000 V
  R6 phys_resid  :     0.19 W
  k=2.0 | F1=0.2146 | P=0.1217 | R=0.9032 | FPR@A6=0.5625
[rule_based] Thresholds calibrated (k=2.5):
  R2 power_delta :     0.07 W
  R3 batt_drain  :     0.39 %/tick
  R5 volt_jump   :   0.0000 V
  R6 phys_resid  :     0.19 W
  k=2.5 | F1=0.2195 | P=0.1256 | R=0.8710 | FPR@A6=0.5417
[rule_based] Thresholds calibrated (k=3.0):
  R2 power_delta :     0.08 W
  R3 batt_drain  :     0.44 %/tick
  R5 volt_jump   :   0.0000 V
  R6 phys_resid  :     0.19 W
  k=3.0 | F1=0.2250 | P=0.1292 | R=0.8710 | FPR@A6=0.5417
[rule_based] Thresholds calibrated (k=3.5):
  R2 power_delta :     0.09 W
  R3 batt_drain  :     0.49 %/tick
  R5 volt_jump   :   0.0000 V
  R6 phys_resid  :     0.19 W
  k=3.5 | F1=0.2338 | P=0.1350 | R=

## Phase 8: Hierarchical Classifier (Level 2)

Random Forest trained on isolated anomalies via SMOTE, evaluating MIR@k (Maintenance-Informed Recall).

In [12]:
!python -m src.models.l2_classifier

[l2] Loading splits …
[l2] Loading LSTM-AE model for L1 predictions …
I0000 00:00:1781744071.858280   22742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781744071.907542   22742 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781744073.118406   22742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1781744073.965541   22804 cuda_executor.cc:1755] Fai

## Phase 9: Explainability (XAI)

Evaluating SHAP feature importance and LSTM reconstruction error mappings.

In [13]:
!python -m src.xai.shap_analysis
!python -m src.xai.reconstruction_analysis

[shap_xai] Loading splits …
[shap_xai] Loaded Isolation Forest: IsolationForest(contamination=0.05, max_features=0.7, n_jobs=-1,
                random_state=42)
[shap_xai] Computing SHAP values for 65 anomalies (background size = 100) ...

=== Phase 9 SHAP Feature Importance (Isolation Forest) ===
  Global Top 5:
curr_a             0.212961
ratio_power_irr    0.145490
ac_power_kw        0.131739
mppt_w_lag1        0.119676
irradiance         0.109738

  Top 3 Features by True Anomaly Type:
    false_data_injection     : ['physics_residual', 'volt_v_lag1', 'volt_v_delta']
    low_irradiance           : ['curr_a', 'ratio_power_irr', 'ac_power_kw']
    offline                  : ['irradiance', 'ratio_power_irr', 'irradiance_lag1']
    sudden_drop              : ['volt_v_lag1', 'irradiance', 'rssi']
[recon_xai] Loading splits …
I0000 00:00:1781744084.767281   23103 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-of

## Phase 10: Edge Hardware Profiling

Benchmarking Latency, RAM (tracemalloc), and quantizing LSTM to TFLite INT8 for ESP32 constraints.

In [14]:
!python -m src.evaluation.edge_profiling

[edge] Loading test subset for profiling...
I0000 00:00:1781744092.129476   23267 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781744093.312895   23267 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

[edge] Quantizing LSTM-AE to TFLite (INT8) ...
Saved artifact at '/tmp/tmpegsszg1w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 24, 36), dtype=tf.float32, name='keras_tensor_28')
Output Type:
  TensorSpec(shape=(1, 24, 36), dtype=tf.float32, name=None)
Captures:
  136194473197648: TensorSpec(shape=(), dtype=tf.resource, name=N

## Phase 11 to 12: Statistical Tests & Visualization

Compiling F1 Tables, McNemar Tests, and 5 standard publication plots.

In [15]:
!python -m src.evaluation.statistical_tests
!python -m src.visualization.plots

[stats] Compiling Phase 11 Tables...
[stats] Warning: could not parse if_sweep.csv: 'f1'
[stats] Warning: could not parse lof_sweep.csv: 'f1'

 TABLE I: F1 Score Breakdown per Anomaly Type
    Method normal low_irradiance  offline  sudden_drop  false_data_injection
Rule-Based  0.000          0.000    0.000        0.000                 1.000
   LSTM-AE  0.000          0.000    0.000        0.000                 1.000
   L2 (RF)      -              -    0.000        0.000                 1.000

 TABLE II: Overall Metrics & Deployment Footprint
          Method  Macro_F1  Precision  Recall  FPR@A6  Latency_ms  Peak_RAM_MB  Size_KB
      Rule-Based     0.032      0.018   0.176   0.521         NaN          NaN      NaN
Isolation Forest     0.000      0.000   0.000   0.000       4.541        0.016 1199.348
         LSTM-AE     0.051      0.030   0.176   0.271      60.324        0.112  907.024

 TABLE III: Best Hyperparameters
    Method              Hyperparameters
Rule-Based                

## Final Output Verification

All results are stored in `outputs/`.

In [16]:
import os
for d in ['outputs/results', 'outputs/figures']:
    print(f'\n{d}:')
    print('\n'.join(os.listdir(d)))


outputs/results:
lstm_ae_model.keras
lstm_ae_model_quantized_dynamic.tflite
if_sweep.csv
edge_profiling_results.csv
if_shap_feat_cols.pkl
rule_based_model.json
Table_1_F1_Per_Class.csv
rule_based_model.pkl
l2_confusion_matrix_test.csv
rule_based_results.csv
if_shap_features.npy
Table_2_Overall_Metrics.csv
if_shap_values.npy
lstmae_reconstruction_errors_test.csv
l2_results.csv
l2_model.pkl
lof_sweep.csv
rule_based_k_sweep.csv
classical_ml_results.csv
l2_feature_importances.csv
Table_3_Hyperparams.csv
l2_confusion_matrix_test.npy
lof_model.pkl
lstm_ae_model_meta.json
lstm_ae_results.csv
lstm_ae_model_meta.pkl
isolation_forest_model.pkl

outputs/figures:
Figure_5_Recon_Heatmap.pdf
tanifi_workflow.mermaid
Figure_4_SHAP_Summary.pdf
Figure_2_Time_Series.pdf
Figure_1_ROC_Curves.pdf
Figure_3_F1_BarChart.pdf
